In [1]:
from __future__ import annotations

import argparse
import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import pyvista as pv


LCS_COLUMNS = [f"lcs{i}{j}" for i in range(1, 4) for j in range(1, 4)]




In [3]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Plot the in-plane principal direction whose signed principal "
            "stress has the largest absolute magnitude."
        )
    )
    parser.add_argument("export_dir", type=Path, help="Abaqus export directory")
    parser.add_argument(
        "--surface",
        choices=("top", "bottom", "envelope"),
        default="envelope",
        help="Shell surface selection (default: thickness envelope)",
    )
    parser.add_argument(
        "--stride",
        type=int,
        default=3,
        help="Draw one direction line for every Nth eligible element",
    )
    parser.add_argument(
        "--min-percentile",
        type=float,
        default=10.0,
        help="Hide directions below this absolute-stress percentile",
    )
    parser.add_argument(
        "--line-length",
        type=float,
        default=None,
        help="Direction-line length in model units (default: automatic)",
    )
    parser.add_argument(
        "--deformation-scale",
        type=float,
        default=0.0,
        help="Plot deformed geometry using this displacement scale",
    )
    parser.add_argument(
        "--screenshot",
        type=Path,
        default=None,
        help="Optional PNG output path",
    )
    parser.add_argument(
        "--off-screen",
        action="store_true",
        help="Render without opening an interactive window",
    )
    return parser.parse_args()


def require_files(directory: Path) -> None:
    missing = [
        name
        for name in ("nodes.csv", "elements.csv", "stress.csv", "metadata.json")
        if not (directory / name).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            f"Missing required export files in {directory}: {', '.join(missing)}"
        )


def load_metadata(directory: Path) -> dict:
    with (directory / "metadata.json").open("r", encoding="utf-8") as stream:
        metadata = json.load(stream)
    if int(metadata.get("element_count", 0)) <= 0:
        raise ValueError(
            "The export contains no elements. Re-export the meshed part instance, "
            "not the assembly reference-point instance."
        )
    return metadata


def build_shell_mesh(directory: Path) -> tuple[pv.PolyData, dict[int, int], pd.DataFrame]:
    nodes = pd.read_csv(directory / "nodes.csv")
    elements = pd.read_csv(directory / "elements.csv")
    if nodes.empty or elements.empty:
        raise ValueError("nodes.csv or elements.csv contains no mesh data")

    node_labels = nodes["node_label"].astype(int).to_numpy()
    points = nodes[["x", "y", "z"]].to_numpy(dtype=float)
    label_to_point = {int(label): index for index, label in enumerate(node_labels)}

    faces: list[list[int]] = []
    kept_rows: list[int] = []
    for row_index, row in elements.iterrows():
        node_count = int(row["node_count"])
        if node_count not in (3, 4):
            continue
        labels = [int(row[f"node_{i}"]) for i in range(1, node_count + 1)]
        try:
            point_ids = [label_to_point[label] for label in labels]
        except KeyError as error:
            raise ValueError(f"Element references missing node label {error.args[0]}")
        faces.append([node_count] + point_ids)
        kept_rows.append(row_index)

    if not faces:
        raise ValueError("No S3/S4-compatible shell elements were found")

    flat_faces = np.asarray([value for face in faces for value in face], dtype=np.int64)
    mesh = pv.PolyData(points, flat_faces)
    kept_elements = elements.loc[kept_rows].reset_index(drop=True)
    element_label_to_cell = {
        int(label): index
        for index, label in enumerate(kept_elements["element_label"].astype(int))
    }
    mesh.cell_data["element_label"] = kept_elements["element_label"].astype(int).to_numpy()
    return mesh, element_label_to_cell, nodes


def apply_displacement(
    mesh: pv.PolyData,
    nodes: pd.DataFrame,
    directory: Path,
    scale: float,
) -> None:
    displacement_path = directory / "displacement.csv"
    if scale == 0.0 or not displacement_path.is_file():
        return
    displacement = pd.read_csv(displacement_path)
    if displacement.empty:
        return
    merged = nodes[["node_label"]].merge(
        displacement[["node_label", "U1", "U2", "U3"]],
        on="node_label",
        how="left",
    )
    vectors = merged[["U1", "U2", "U3"]].fillna(0.0).to_numpy(dtype=float)
    mesh.point_data["displacement"] = vectors
    mesh.points = mesh.points + float(scale) * vectors


def choose_shell_surface(stress: pd.DataFrame, surface: str) -> pd.DataFrame:
    description = stress["section_point_description"].fillna("").astype(str).str.upper()
    if surface == "top":
        mask = description.str.contains("SPOS", regex=False)
    elif surface == "bottom":
        mask = description.str.contains("SNEG", regex=False)
    elif surface == "envelope":
        mask = description.str.contains("SPOS", regex=False) | description.str.contains(
            "SNEG", regex=False
        )
    else:
        raise ValueError("surface must be 'top', 'bottom', or 'envelope'")
    selected = stress[mask].copy()
    if not selected.empty:
        return selected

    # Some ODB versions omit SPOS/SNEG text. Fall back to the outer section
    # point numbers independently for every element and integration point.
    section_number = pd.to_numeric(stress["section_point_number"], errors="coerce")
    stress = stress.copy()
    stress["_section_number"] = section_number
    group_columns = ["element_label", "integration_point"]
    groups = stress.groupby(group_columns)["_section_number"]
    minimum = groups.transform("min")
    maximum = groups.transform("max")
    if surface == "top":
        mask = stress["_section_number"] == maximum
    elif surface == "bottom":
        mask = stress["_section_number"] == minimum
    else:
        mask = (stress["_section_number"] == minimum) | (
            stress["_section_number"] == maximum
        )
    selected = stress[mask].copy()
    if selected.empty:
        raise ValueError(f"Could not identify the {surface} shell section point")
    return selected


def cell_normal(mesh: pv.PolyData, cell_id: int) -> np.ndarray:
    ids = np.asarray(mesh.get_cell(cell_id).point_ids, dtype=int)
    points = np.asarray(mesh.points[ids], dtype=float)
    normal = np.cross(points[1] - points[0], points[2] - points[0])
    norm = np.linalg.norm(normal)
    if norm <= 1.0e-14:
        return np.array([0.0, 0.0, 1.0])
    return normal / norm


def local_to_global_direction(
    local_direction: np.ndarray,
    lcs: np.ndarray,
    normal: np.ndarray,
    lcs_present: bool,
) -> np.ndarray:
    if lcs_present and np.all(np.isfinite(lcs)):
        # Abaqus ODB releases differ in how users interpret the stored direction
        # cosine matrix. Both conventions are tested; a shell principal direction
        # must be tangent, so retain the candidate closest to the element plane.
        candidates = (lcs @ local_direction, lcs.T @ local_direction)
        direction = min(candidates, key=lambda vector: abs(float(np.dot(vector, normal))))
    else:
        # Reconstruct a tangent basis as a safe fallback. This branch should be
        # rare because shell stress FieldValues normally provide an LCS.
        trial = np.array([1.0, 0.0, 0.0])
        if abs(float(np.dot(trial, normal))) > 0.9:
            trial = np.array([0.0, 1.0, 0.0])
        e1 = trial - np.dot(trial, normal) * normal
        e1 /= np.linalg.norm(e1)
        e2 = np.cross(normal, e1)
        direction = local_direction[0] * e1 + local_direction[1] * e2

    # Remove numerical normal leakage and normalize. Direction is signless.
    direction = direction - np.dot(direction, normal) * normal
    norm = np.linalg.norm(direction)
    if norm <= 1.0e-14:
        return np.array([1.0, 0.0, 0.0])
    return direction / norm


def compute_element_principal_data(
    mesh: pv.PolyData,
    element_label_to_cell: dict[int, int],
    stress: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    cell_count = mesh.n_cells
    abs_principal = np.full(cell_count, np.nan)
    signed_principal = np.full(cell_count, np.nan)
    directions = np.full((cell_count, 3), np.nan)
    centers = np.asarray(mesh.cell_centers().points)

    required = {"S11", "S22", "S12"}
    missing = sorted(required.difference(stress.columns))
    if missing:
        raise ValueError(f"stress.csv is missing components: {', '.join(missing)}")

    # S4R normally has one integration point. If several exist, select the row
    # with the largest governing absolute in-plane principal stress per element.
    for element_label, rows in stress.groupby("element_label", sort=False):
        cell_id = element_label_to_cell.get(int(element_label))
        if cell_id is None:
            continue
        normal = cell_normal(mesh, cell_id)
        best = None
        for _, row in rows.iterrows():
            tensor_2d = np.array(
                [[float(row["S11"]), float(row["S12"])],
                 [float(row["S12"]), float(row["S22"])]],
                dtype=float,
            )
            eigenvalues, eigenvectors = np.linalg.eigh(tensor_2d)
            principal_index = int(np.argmax(np.abs(eigenvalues)))
            signed_value = float(eigenvalues[principal_index])
            local_2d = eigenvectors[:, principal_index]
            local_3d = np.array([local_2d[0], local_2d[1], 0.0])
            lcs = row[LCS_COLUMNS].to_numpy(dtype=float).reshape(3, 3)
            global_direction = local_to_global_direction(
                local_3d,
                lcs,
                normal,
                bool(int(row.get("lcs_present", 0))),
            )
            candidate = (abs(signed_value), signed_value, global_direction)
            if best is None or candidate[0] > best[0]:
                best = candidate
        if best is not None:
            abs_principal[cell_id], signed_principal[cell_id], directions[cell_id] = best

    valid = np.isfinite(abs_principal) & np.all(np.isfinite(directions), axis=1)
    if not np.any(valid):
        raise ValueError("No valid principal-stress directions could be calculated")
    return abs_principal, signed_principal, directions, centers


def make_direction_lines(
    centers: np.ndarray,
    directions: np.ndarray,
    values: np.ndarray,
    eligible: np.ndarray,
    length: float,
) -> pv.PolyData:
    ids = np.flatnonzero(eligible)
    half = 0.5 * float(length)
    start = centers[ids] - half * directions[ids]
    end = centers[ids] + half * directions[ids]
    points = np.empty((2 * len(ids), 3), dtype=float)
    points[0::2] = start
    points[1::2] = end
    lines = np.column_stack(
        [
            np.full(len(ids), 2, dtype=np.int64),
            2 * np.arange(len(ids), dtype=np.int64),
            2 * np.arange(len(ids), dtype=np.int64) + 1,
        ]
    ).ravel()
    # Construct an empty PolyData first. pv.PolyData(points) creates one vertex
    # cell per point; retaining those vertex cells would make cell-data length
    # differ from the number of direction lines.
    poly = pv.PolyData()
    poly.points = points
    poly.lines = lines
    poly.cell_data["abs_principal"] = values[ids]
    return poly


def automatic_line_length(mesh: pv.PolyData) -> float:
    diagonal = float(mesh.length)
    return 0.9 * diagonal / max(np.sqrt(float(mesh.n_cells)), 1.0)


def show_scalar_map(
    mesh: pv.PolyData,
    scalar_name: str,
    title: str,
    *,
    cmap: str = "turbo",
    deformation: np.ndarray | None = None,
    deformation_scale: float = 0.0,
) -> None:
    plot_mesh = mesh.copy(deep=True)
    if deformation is not None and deformation_scale != 0.0:
        plot_mesh.points = plot_mesh.points + float(deformation_scale) * deformation
    plotter = pv.Plotter()
    plotter.set_background("white")
    plotter.add_mesh(
        plot_mesh,
        scalars=scalar_name,
        cmap=cmap,
        show_edges=True,
        edge_color="#555555",
        line_width=0.3,
        nan_color="lightgray",
        scalar_bar_args={"title": title, "color": "black"},
    )
    plotter.add_axes(color="black")
    plotter.view_isometric()
    plotter.camera.zoom(1.2)
    plotter.show()


def load_nodal_displacement(
    directory: Path, nodes: pd.DataFrame
) -> np.ndarray:
    path = directory / "displacement.csv"
    if not path.is_file():
        raise FileNotFoundError(f"Missing displacement results: {path}")
    displacement = pd.read_csv(path)
    required = {"node_label", "U1", "U2", "U3"}
    missing = sorted(required.difference(displacement.columns))
    if missing:
        raise ValueError(f"displacement.csv is missing: {', '.join(missing)}")
    merged = nodes[["node_label"]].merge(
        displacement[["node_label", "U1", "U2", "U3"]],
        on="node_label",
        how="left",
    )
    return merged[["U1", "U2", "U3"]].fillna(0.0).to_numpy(dtype=float)


def plot_displacement_map(
    export_dir,
    component: str = "magnitude",
    direction=None,
    deformation_scale: float = 0.0,
) -> None:
    """Plot U magnitude, U1/U2/U3, or displacement along a vector."""
    directory = Path(export_dir).expanduser().resolve()
    mesh, _, nodes = build_shell_mesh(directory)
    displacement = load_nodal_displacement(directory, nodes)
    mesh.point_data["U_vector"] = displacement

    key = component.lower()
    if key == "magnitude":
        values = np.linalg.norm(displacement, axis=1)
        name, title = "U_magnitude", "Displacement magnitude (mm)"
    elif key in ("u1", "u2", "u3"):
        index = {"u1": 0, "u2": 1, "u3": 2}[key]
        values = displacement[:, index]
        name, title = key.upper(), f"{key.upper()} displacement (mm)"
    elif key == "direction":
        if direction is None:
            raise ValueError("Provide direction=(x, y, z) for component='direction'")
        vector = np.asarray(direction, dtype=float).reshape(3)
        norm = np.linalg.norm(vector)
        if norm <= 1.0e-12:
            raise ValueError("direction cannot be the zero vector")
        vector /= norm
        values = displacement @ vector
        name = "U_direction"
        title = f"Displacement along ({vector[0]:.2f}, {vector[1]:.2f}, {vector[2]:.2f}) (mm)"
    else:
        raise ValueError("component must be magnitude, U1, U2, U3, or direction")

    mesh.point_data[name] = values
    governing = int(np.nanargmax(np.abs(values)))
    print(f"Governing value: {values[governing]:.6g} mm")
    print(f"Node label: {int(nodes.iloc[governing]['node_label'])}")
    print(f"Coordinates: {mesh.points[governing]}")
    show_scalar_map(
        mesh,
        name,
        title,
        deformation=displacement,
        deformation_scale=deformation_scale,
    )


def plot_stress_map(
    export_dir,
    quantity: str = "mises",
    surface: str = "envelope",
) -> None:
    """Plot Mises or in-plane principal stress on top/bottom/envelope."""
    directory = Path(export_dir).expanduser().resolve()
    mesh, element_label_to_cell, _ = build_shell_mesh(directory)
    stress = pd.read_csv(directory / "stress.csv")
    if stress.empty:
        raise ValueError("stress.csv contains no values")
    stress = choose_shell_surface(stress, surface)
    values = np.full(mesh.n_cells, np.nan)

    for element_label, rows in stress.groupby("element_label", sort=False):
        cell_id = element_label_to_cell.get(int(element_label))
        if cell_id is None:
            continue
        if quantity == "mises":
            value = rows["mises"].max()
        elif quantity == "max_principal":
            value = rows["max_in_plane_principal"].max()
        elif quantity == "min_principal":
            value = rows["min_in_plane_principal"].min()
        elif quantity == "abs_principal":
            candidates = np.concatenate(
                [
                    rows["max_in_plane_principal"].to_numpy(dtype=float),
                    rows["min_in_plane_principal"].to_numpy(dtype=float),
                ]
            )
            value = candidates[int(np.nanargmax(np.abs(candidates)))]
        else:
            raise ValueError(
                "quantity must be mises, max_principal, min_principal, or abs_principal"
            )
        values[cell_id] = float(value)

    labels = {
        "mises": ("von_mises", "Von Mises stress (MPa)"),
        "max_principal": ("max_principal", "Maximum in-plane principal stress (MPa)"),
        "min_principal": ("min_principal", "Minimum in-plane principal stress (MPa)"),
        "abs_principal": ("abs_principal", "Governing absolute principal stress (MPa)"),
    }
    name, title = labels[quantity]
    mesh.cell_data[name] = values
    if quantity in ("min_principal", "abs_principal"):
        governing = int(np.nanargmax(np.abs(values)))
    else:
        governing = int(np.nanargmax(values))
    print(f"Surface selection: {surface}")
    print(f"Governing value: {values[governing]:.6g} MPa")
    print(f"Element label: {int(mesh.cell_data['element_label'][governing])}")
    show_scalar_map(mesh, name, f"{title} — {surface}")


def plot_principal_directions(
    export_dir,
    surface: str = "envelope",
    stride: int = 3,
    min_percentile: float = 10.0,
    line_length: float | None = None,
    deformation_scale: float = 0.0,
    screenshot=None,
    off_screen: bool = False,
) -> None:
    """Plot the absolute governing in-plane principal direction field."""
    args = SimpleNamespace(
        export_dir=Path(export_dir),
        surface=surface,
        stride=stride,
        min_percentile=min_percentile,
        line_length=line_length,
        deformation_scale=deformation_scale,
        screenshot=None if screenshot is None else Path(screenshot),
        off_screen=off_screen,
    )
    directory = args.export_dir.expanduser().resolve()
    require_files(directory)
    metadata = load_metadata(directory)
    mesh, element_label_to_cell, nodes = build_shell_mesh(directory)
    apply_displacement(mesh, nodes, directory, args.deformation_scale)

    stress = pd.read_csv(directory / "stress.csv")
    if stress.empty:
        raise ValueError("stress.csv contains no values")
    stress = choose_shell_surface(stress, args.surface)
    abs_principal, signed_principal, directions, centers = compute_element_principal_data(
        mesh, element_label_to_cell, stress
    )
    mesh.cell_data["abs_governing_principal"] = abs_principal
    mesh.cell_data["signed_governing_principal"] = signed_principal
    mesh.cell_data["principal_direction"] = directions

    valid_values = abs_principal[np.isfinite(abs_principal)]
    threshold = float(np.percentile(valid_values, args.min_percentile))
    eligible = np.isfinite(abs_principal) & (abs_principal >= threshold)
    stride = max(int(args.stride), 1)
    eligible &= (np.arange(mesh.n_cells) % stride) == 0
    line_length = (
        float(args.line_length)
        if args.line_length is not None
        else automatic_line_length(mesh)
    )
    direction_lines = make_direction_lines(
        centers, directions, abs_principal, eligible, line_length
    )

    plotter = pv.Plotter(off_screen=args.off_screen)
    plotter.set_background("white")
    plotter.add_mesh(
        mesh,
        scalars="abs_governing_principal",
        cmap="turbo",
        show_edges=True,
        edge_color="#5b6472",
        line_width=0.35,
        nan_color="lightgray",
        scalar_bar_args={
            "title": "|Governing in-plane principal stress| (MPa)",
            "color": "black",
            "title_font_size": 15,
            "label_font_size": 12,
        },
    )
    plotter.add_mesh(
        direction_lines,
        color="black",
        line_width=2.2,
        render_lines_as_tubes=True,
    )
    plotter.add_axes(color="black")
    plotter.add_text(
        f"Absolute governing principal directions — {args.surface.upper()}\n"
        f"{metadata.get('instance', '')} | step {metadata.get('step', '')}",
        position="upper_left",
        color="black",
        font_size=11,
    )
    plotter.view_isometric()
    plotter.camera.zoom(1.25)

    if args.screenshot is not None:
        screenshot = args.screenshot.expanduser().resolve()
        screenshot.parent.mkdir(parents=True, exist_ok=True)
        plotter.show(screenshot=str(screenshot), auto_close=args.off_screen)
        print(f"Saved screenshot: {screenshot}")
    elif args.off_screen:
        raise ValueError("--off-screen requires --screenshot")
    else:
        plotter.show()

    print(f"Surface: {args.surface}")
    print(f"Elements with stress: {np.count_nonzero(np.isfinite(abs_principal))}")
    print(f"Direction lines shown: {direction_lines.n_cells}")
    print(f"Maximum absolute governing principal stress: {np.nanmax(abs_principal):.6g} MPa")


def main() -> None:
    """Command-line entry point; notebook users call plotting functions directly."""
    args = parse_args()
    plot_principal_directions(
        args.export_dir,
        surface=args.surface,
        stride=args.stride,
        min_percentile=args.min_percentile,
        line_length=args.line_length,
        deformation_scale=args.deformation_scale,
        screenshot=args.screenshot,
        off_screen=args.off_screen,
    )

In [10]:
# Change only this path if your export folder is elsewhere.
EXPORT_DIR = Path(r"/home/arash/HVD_SeedsBase/AbaqusData/Test1_Full_pyvista_export")

plot_principal_directions(
    EXPORT_DIR,
    surface="envelope",
    stride=1,
    min_percentile=5.0,
)

Widget(value='<iframe src="http://localhost:43815/index.html?ui=P_0x75b6711baf20_9&reconnect=auto" class="pyvi…

Surface: envelope
Elements with stress: 2150
Direction lines shown: 2042
Maximum absolute governing principal stress: 53.9521 MPa


In [5]:
plot_displacement_map(
    EXPORT_DIR,
    component="magnitude",
)
plot_stress_map(
    EXPORT_DIR,
    quantity="mises",
    surface="envelope",
)

Governing value: 0.5148 mm
Node label: 2194
Coordinates: [ 0.73745185 -1.28519022  9.95108891]


Widget(value='<iframe src="http://localhost:43815/index.html?ui=P_0x75b6d0635450_1&reconnect=auto" class="pyvi…

Surface selection: envelope
Governing value: 51.1473 MPa
Element label: 1


Widget(value='<iframe src="http://localhost:43815/index.html?ui=P_0x75b6f72eb2b0_2&reconnect=auto" class="pyvi…

In [6]:
plot_displacement_map(
    EXPORT_DIR,
    component="direction",
    direction=(1, 0, 0),
)

Governing value: -0.084202 mm
Node label: 1429
Coordinates: [ 5.43056631 -1.28513038 10.45115948]


Widget(value='<iframe src="http://localhost:43815/index.html?ui=P_0x75b6d2966050_3&reconnect=auto" class="pyvi…

In [7]:
plot_stress_map(EXPORT_DIR, quantity="max_principal", surface="envelope")
plot_stress_map(EXPORT_DIR, quantity="min_principal", surface="envelope")
plot_stress_map(EXPORT_DIR, quantity="abs_principal", surface="envelope")

Surface selection: envelope
Governing value: 53.9521 MPa
Element label: 1


Widget(value='<iframe src="http://localhost:43815/index.html?ui=P_0x75b6d25efb20_4&reconnect=auto" class="pyvi…

Surface selection: envelope
Governing value: -37.385 MPa
Element label: 5


Widget(value='<iframe src="http://localhost:43815/index.html?ui=P_0x75b6f72b1fc0_5&reconnect=auto" class="pyvi…

Surface selection: envelope
Governing value: 53.9521 MPa
Element label: 1


Widget(value='<iframe src="http://localhost:43815/index.html?ui=P_0x75b6d0637f70_6&reconnect=auto" class="pyvi…